In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import classification_report
import nltk
import numpy as np
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
nltk.download('punkt_tab')
from transformers.pipelines.pt_utils import KeyDataset




[nltk_data] Downloading package punkt to
[nltk_data]     /ihome/xli/sek188/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /ihome/xli/sek188/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
odf = pd.read_csv("annotations.csv", encoding="utf-8")

In [3]:
odf.shape

(17775, 24)

In [6]:
# fdf = odf.sample(1000, random_state=42)
fdf = odf.drop_duplicates(subset=["sentence_id"]).copy()
fdf["clean"] = (
    "\n ARTICLE TEXT: \n" +
    fdf["text"].astype(str)
)

In [7]:

model_name = "mediabiasgroup/da-roberta-babe-ft"
tokenizer = AutoTokenizer.from_pretrained(model_name)
MODEL_NAME = AutoModelForSequenceClassification.from_pretrained(model_name)


# classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

classifier = pipeline(
    "text-classification",
    model=MODEL_NAME,
    device=0,                   
    return_all_scores=True,     
    truncation=True,
    tokenizer=tokenizer
)

# text = "the left gets absolutely roasted by the super awesome gigachad trump" lable 1, score 0.938
text = "the fed cuts interest rates" #label 0, score 0.95
result = classifier(text)

print(result)

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Device set to use cuda:0
/ihome/xli/sek188/MSTHESIS/envs/.venv/lib/python3.11/site-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


[[{'label': 'LABEL_0', 'score': 0.9565666913986206}, {'label': 'LABEL_1', 'score': 0.04343331232666969}]]


In [8]:
LABEL_POS = "LABEL_1"  # set to your positive/target label
LABEL_NEG = "LABEL_0"

def classify_article(text):
    # 1) split into sentences
    sentences = [s for s in sent_tokenize(text or "") if s.strip()]
    if not sentences:
        return {
            "label": "error",
            "score": None,
            "scores": None,
            "adjusted_score": None,
            "error": "No sentences after tokenization."
        }

    try:
        # 2) run classifier in batch over all sentences
        # each element of `results` is a list of dicts: [{'label': 'LABEL_0','score':...}, {'label':'LABEL_1','score':...}]
        results = classifier(
            sentences,
            batch_size=32,        # <-- helps avoid the "sequentially on GPU" warning
            truncation=True,
            padding=True
        )

        # defensive: ensure consistent structure
        if not isinstance(results, list) or not results or not isinstance(results[0], list):
            return {
                "label": "error",
                "score": None,
                "scores": None,
                "adjusted_score": None,
                "error": f"Unexpected pipeline output structure: {type(results)}"
            }

        # 3) build a [num_sentences, num_classes] score array in a consistent label order
        # use labels from the first row as the canonical order
        label_order = [d["label"] for d in results[0]]
        score_array = np.array([[d["score"] for d in row] for row in results])  # shape: [S, C]
        mean_scores = score_array.mean(axis=0)                                   # shape: [C]

        # 4) adjusted score: POS - NEG averaged over sentences
        def get_idx(lbl):
            try:
                return label_order.index(lbl)
            except ValueError:
                return None

        pos_idx = get_idx(LABEL_POS)
        neg_idx = get_idx(LABEL_NEG)
        if pos_idx is not None and neg_idx is not None:
            adjusted_score = float(mean_scores[pos_idx] - mean_scores[neg_idx])
        else:
            adjusted_score = None

        # 5) final label (pick the higher of mean POS vs NEG when both exist, else argmax)
        if pos_idx is not None and neg_idx is not None:
            pred_label = LABEL_POS if mean_scores[pos_idx] >= mean_scores[neg_idx] else LABEL_NEG
        else:
            pred_label = label_order[int(np.argmax(mean_scores))]

        return {
            "label": pred_label,
            "score": mean_scores,     # np.array of per-class means in `label_order`
            "scores": score_array,    # np.array of per-sentence, per-class
            "adjusted_score": adjusted_score,
            "labels": label_order
        }

    except Exception as e:
        # don't reference `results` here—may be undefined if pipeline threw
        return {
            "label": "error",
            "score": None,
            "scores": None,
            "adjusted_score": None,
            "error": str(e)
        }


In [9]:
# fdf = fdf.sample(2)
fdf['spinde_output'] = fdf['clean'].apply(classify_article)

fdf['spinde_label'] = fdf['spinde_output'].apply(lambda x: x['label'])
fdf['spinde_score'] = fdf['spinde_output'].apply(lambda x: x['score'])
fdf['spinde_scores'] = fdf['spinde_output'].apply(lambda x: x['scores'])

# # This yields one pipeline output per row in fdf["clean"]
# all_outputs = []
# for out in classifier(KeyDataset(fdf, "clean"), batch_size=32, truncation=True, padding=True):
#     all_outputs.append(out)



# [{'label': 'LABEL_0', 'score': 0.9084169268608093}, {'label': 'LABEL_0', 'score': 0.9547224640846252}]
# string indices must be integers
# [{'label': 'LABEL_0', 'score': 0.9597590565681458}]

# string indices must be integers

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [22]:
# wdf = wdf.reset_index()
# res = wdf.loc[0, 'spinde_output']

# wdf.head()
# wdf.to_csv('spinde_wide_form_jul30.csv')


fdf.head()


,Unnamed: 0.1,Unnamed: 0,survey_record_id,sentence_id,sentence_group_id,created_at,label,words,factual,group_id,...,native_english_speaker,political_ideology,followed_news_outlets,news_check_frequency,survey_completed,clean,spinde_output,spinde_label,spinde_score,spinde_scores
5115,6352,22431,49fab1bae21e4bcea965df039c139a66,e8c98a293b3644cdb8ab313d9f8c8511,25,2020-08-13 21:48:47,Biased,"serious ,warned ,dire",Expresses writer’s opinion,25,...,Native speaker,5,"['Fox News', 'The Wall Street Journal', 'ABC N...",Every day,True,\n ARTICLE TEXT: \nTheir work came on the heel...,"{'label': 'LABEL_0', 'score': [0.8983498215675...",LABEL_0,"[0.8983498215675354, 0.1016501635313034]","[[0.8983498215675354, 0.1016501635313034]]"
5605,7002,18901,50f63b27dacd4e9eb5646ee3c15521dd,54805c95a9ff490d86c31a3551a6ff08,81,2020-08-12 02:37:19,Non-biased,appears,Entirely factual,81,...,Native speaker,-7,"['New York Times', 'Reuters', 'CNN', 'The Wash...",Several times per day,True,\n ARTICLE TEXT: \nThe GOP controls both the l...,"{'label': 'LABEL_0', 'score': [0.8883294463157...",LABEL_0,"[0.8883294463157654, 0.11167050153017044]","[[0.8883294463157654, 0.11167050153017044]]"
17709,22405,21889,feda5e95e6114340b9d0eb945f6c626c,ad9c16845c6b48acb039300f0d407236,71,2020-08-13 15:48:10,Biased,"aggressive,worried,sounding fresh alarms",Somewhat factual but also opinionated,71,...,Native speaker,3,"['CNN', 'The Wall Street Journal', 'Bloomberg']",Every day,True,\n ARTICLE TEXT: \nAuthoritarianism experts wh...,"{'label': 'LABEL_0', 'score': [0.5238929986953...",LABEL_0,"[0.5238929986953735, 0.4761069715023041]","[[0.5238929986953735, 0.4761069715023041]]"
1557,2054,15327,189a99e07c5e4b03b365e0c825b13e29,d74e5da91b7644e1b792f926c02e52fd,62,2020-08-09 00:38:58,Biased,"little impact,time in the world",Expresses writer’s opinion,62,...,Native speaker,-5,"['Fox News', 'CNN']",Several times per week,True,\n ARTICLE TEXT: \nSuch a ban would have been ...,"{'label': 'LABEL_1', 'score': [0.3730577826499...",LABEL_1,"[0.3730577826499939, 0.6269421577453613]","[[0.3730577826499939, 0.6269421577453613]]"
16506,20882,17732,ee7da49a4a1542c2afd46dab46c5db2d,a3fef07c591a4a08884e2cba937e986a,81,2020-08-09 16:38:38,Biased,"irresponsible,handouts",Somewhat factual but also opinionated,81,...,Native speaker,-8,"['Reuters', 'New York Times', 'The Guardian', ...",Every day,True,"\n ARTICLE TEXT: \nAt the time, Moynihan was p...","{'label': 'LABEL_1', 'score': [0.1919542402029...",LABEL_1,"[0.19195424020290375, 0.8080458045005798]","[[0.19195424020290375, 0.8080458045005798]]"


In [10]:
# cdf.head()
fdf.to_csv("w23_bias_spinde.csv")